In [3]:
from google.colab import drive
import os
import pandas as pd
import numpy as np
import duckdb


In [4]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
os.listdir('/content/drive/MyDrive/MIMIC-IV')

['icustays.csv.gz',
 'chartevents.csv.gz',
 'patients.csv.gz',
 'admissions.csv.gz',
 'd_items.csv.gz']

#####start with looking at the head of all the smaller files but chartevents
#####generating the csvs for all the smaller files: d_items,icustays,patients and admissions

In [6]:
d_items_path='/content/drive/MyDrive/MIMIC-IV/d_items.csv.gz'
icustays_path='/content/drive/MyDrive/MIMIC-IV/icustays.csv.gz'
patients_path='/content/drive/MyDrive/MIMIC-IV/patients.csv.gz'
admissions_path='/content/drive/MyDrive/MIMIC-IV/admissions.csv.gz'
chartevents_path='/content/drive/MyDrive/MIMIC-IV/chartevents.csv.gz'
d_items=pd.read_csv(d_items_path)
icustays=pd.read_csv(icustays_path)
patients=pd.read_csv(patients_path)
admissions=pd.read_csv(admissions_path)
print("all the files have been converted to csvs")

all the files have been converted to csvs


In [7]:
d_items.head()

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,220001,Problem List,Problem List,chartevents,General,NaN,Text,NaN,NaN
1,220003,ICU Admission date,ICU Admission date,datetimeevents,ADT,NaN,Date and time,NaN,NaN
2,220045,Heart Rate,HR,chartevents,Routine Vital Signs,bpm,Numeric,NaN,NaN
3,220046,Heart rate Alarm - High,HR Alarm - High,chartevents,Alarms,bpm,Numeric,NaN,NaN
4,220047,Heart Rate Alarm - Low,HR Alarm - Low,chartevents,Alarms,bpm,Numeric,NaN,NaN


<h5>inferences: everything except item id, label and abbrev is pretty insignificant so just the first 3 is sufficient for now and check which is most linked to preferably its chartevents</h5>



In [8]:
patients.head()

,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10000032,F,52,2180,2014 - 2016,2180-09-09
1,10000048,F,23,2126,2008 - 2010,NaN
2,10000058,F,33,2168,2020 - 2022,NaN
3,10000068,F,19,2160,2008 - 2010,NaN
4,10000084,M,72,2160,2017 - 2019,2161-02-13


<h5>inferences: subject id will be the only fruitful column in this table and will link to icustays.</h5>



In [9]:
icustays.head()

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252
2,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535
3,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032
4,10001217,27703517,34592300,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-12-19 15:42:24,2157-12-20 14:27:41,0.948113


<h5>as said above stay id is linked to subj id here but intime and outtime is pretty useful from here.</h5>



In [10]:
admissions.head()

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,English,WIDOWED,WHITE,2180-05-06 19:17:00,2180-05-06 23:30:00,0
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaN,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-06-26 15:54:00,2180-06-26 21:31:00,0
2,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaN,EW EMER.,P19UTS,EMERGENCY ROOM,HOSPICE,Medicaid,English,WIDOWED,WHITE,2180-08-05 20:58:00,2180-08-06 01:44:00,0
3,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,NaN,EW EMER.,P06OTX,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-07-23 05:54:00,2180-07-23 14:00:00,0
4,10000068,25022803,2160-03-03 23:16:00,2160-03-04 06:26:00,NaN,EU OBSERVATION,P39NWO,EMERGENCY ROOM,NaN,NaN,English,SINGLE,WHITE,2160-03-03 21:55:00,2160-03-04 06:26:00,0


<h5>inferences: not completely sure of the distinction between admittime, intime etc check doc and almost everything but the first few cols is redundant</h5>



In [11]:
# for chartevents alone:
chartevents_preview=pd.read_csv('/content/drive/MyDrive/MIMIC-IV/chartevents.csv.gz',nrows=5)

chartevents_preview.head()

,subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valuenum,valueuom,warning
0,10000032,29079034,39553978,18704,2180-07-23 12:36:00,2180-07-23 14:45:00,226512,39.4,39.4,kg,0
1,10000032,29079034,39553978,18704,2180-07-23 12:36:00,2180-07-23 14:45:00,226707,60,60.0,Inch,0
2,10000032,29079034,39553978,18704,2180-07-23 12:36:00,2180-07-23 14:45:00,226730,152,152.0,cm,0
3,10000032,29079034,39553978,18704,2180-07-23 14:00:00,2180-07-23 14:18:00,220048,SR (Sinus Rhythm),NaN,NaN,0
4,10000032,29079034,39553978,18704,2180-07-23 14:00:00,2180-07-23 14:18:00,224642,Oral,NaN,NaN,0


<h4>most important one linking all of them, need to take batches of data based on the subject id and then cumulate all the values from the items</h4>
<h5> itemid from d_items must be subs here and grouping done by subject_id;</h5>



In [12]:
print("patients columns:",patients.columns.tolist())
print("icustays columns:",icustays.columns.tolist())
print("admissions columns:",admissions.columns.tolist())
print("d_items columns:",d_items.columns.tolist())
print("chartevents columns:",chartevents_preview.columns.tolist())

patients columns: ['subject_id', 'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod']
icustays columns: ['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit', 'intime', 'outtime', 'los']
admissions columns: ['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'admit_provider_id', 'admission_location', 'discharge_location', 'insurance', 'language', 'marital_status', 'race', 'edregtime', 'edouttime', 'hospital_expire_flag']
d_items columns: ['itemid', 'label', 'abbreviation', 'linksto', 'category', 'unitname', 'param_type', 'lownormalvalue', 'highnormalvalue']
chartevents columns: ['subject_id', 'hadm_id', 'stay_id', 'caregiver_id', 'charttime', 'storetime', 'itemid', 'value', 'valuenum', 'valueuom', 'warning']


In [13]:
#to create a a dictionary with most of the important vitals there are
print(d_items[d_items['label'].str.contains('heart rate', case=False, na=False)])
print("-------------------------------")
print(d_items[d_items['label'].str.contains('respiratory rate', case=False, na=False)])
print("-------------------------------")
print(d_items[d_items['label'].str.contains('systolic', case=False, na=False)])
print("-------------------------------")
print(d_items[d_items['label'].str.contains('temperature', case=False, na=False)])
print("-------------------------------")
print(d_items[d_items['label'].str.contains('diastolic', case=False, na=False)])
print("-------------------------------")
print(d_items[d_items['label'].str.contains('oxygen', case=False, na=False)])


   itemid                    label     abbreviation      linksto  \
2  220045               Heart Rate               HR  chartevents   
3  220046  Heart rate Alarm - High  HR Alarm - High  chartevents   
4  220047   Heart Rate Alarm - Low   HR Alarm - Low  chartevents   

              category unitname param_type  lownormalvalue  highnormalvalue  
2  Routine Vital Signs      bpm    Numeric             NaN              NaN  
3               Alarms      bpm    Numeric             NaN              NaN  
4               Alarms      bpm    Numeric             NaN              NaN  
-------------------------------
     itemid                           label                    abbreviation  \
28   220210                Respiratory Rate                              RR   
799  224688          Respiratory Rate (Set)          Respiratory Rate (Set)   
800  224689  Respiratory Rate (spontaneous)  Respiratory Rate (spontaneous)   
801  224690        Respiratory Rate (Total)        Respiratory Rate

In [14]:
#mapping dictionary for the vitals from d_items file:
vitals_dict={"spO2":220277,"temp(C)":223762,"heartrate":220045,"ARTsys":225309,"ARTdia":225310,"ARTmean":225312,"NBPs":220179,"NBPd":220180,"NBPm":220181,"RR":224688}
vitals,ids=list(vitals_dict.keys()),list(vitals_dict.values())
vitals_rev_dict={vitals_dict[i]:i for i in vitals_dict}
print(vitals,ids)

['spO2', 'temp(C)', 'heartrate', 'ARTsys', 'ARTdia', 'ARTmean', 'NBPs', 'NBPd', 'NBPm', 'RR'] [220277, 223762, 220045, 225309, 225310, 225312, 220179, 220180, 220181, 224688]


In [15]:
import duckdb

In [16]:
con=duckdb.connect()
queryrows=f"""
SELECT subject_id, stay_id, charttime, itemid, value, valuenum
FROM read_csv_auto('{chartevents_path}')
WHERE itemid IN {tuple(ids)}
LIMIT 25
"""
result=con.execute(queryrows).fetchdf()
result

,subject_id,stay_id,charttime,itemid,value,valuenum
0,10000032,39553978,2180-07-23 14:11:00,220179,84,84.0
1,10000032,39553978,2180-07-23 14:11:00,220180,48,48.0
2,10000032,39553978,2180-07-23 14:11:00,220181,56,56.0
3,10000032,39553978,2180-07-23 14:12:00,220045,91,91.0
4,10000032,39553978,2180-07-23 14:13:00,220277,98,98.0
5,10000032,39553978,2180-07-23 14:30:00,220045,93,93.0
6,10000032,39553978,2180-07-23 14:30:00,220179,95,95.0
7,10000032,39553978,2180-07-23 14:30:00,220180,59,59.0
8,10000032,39553978,2180-07-23 14:30:00,220181,67,67.0
9,10000032,39553978,2180-07-23 14:30:00,220277,97,97.0


In [18]:
result["vitals"] = result["itemid"].map(vitals_rev_dict)

result

,subject_id,stay_id,charttime,itemid,value,valuenum,vital,vitals
0,10000032,39553978,2180-07-23 14:11:00,220179,84,84.0,NBPs,NBPs
1,10000032,39553978,2180-07-23 14:11:00,220180,48,48.0,NBPd,NBPd
2,10000032,39553978,2180-07-23 14:11:00,220181,56,56.0,NBPm,NBPm
3,10000032,39553978,2180-07-23 14:12:00,220045,91,91.0,heartrate,heartrate
4,10000032,39553978,2180-07-23 14:13:00,220277,98,98.0,spO2,spO2
5,10000032,39553978,2180-07-23 14:30:00,220045,93,93.0,heartrate,heartrate
6,10000032,39553978,2180-07-23 14:30:00,220179,95,95.0,NBPs,NBPs
7,10000032,39553978,2180-07-23 14:30:00,220180,59,59.0,NBPd,NBPd
8,10000032,39553978,2180-07-23 14:30:00,220181,67,67.0,NBPm,NBPm
9,10000032,39553978,2180-07-23 14:30:00,220277,97,97.0,spO2,spO2
